## Assignment 1

### Dheeraj Ippakayal
- 2602401280013

### Shree Koshti
- 2602401280043

In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
### 1st Part

## 2

# a
input_file = f"/home/talentum/Desktop/Assignments/Assignment1/Pair-RDD/selfishgiant.txt"

splitRdd = sc.textFile(f"FILE://{input_file}").flatMap(lambda line: line.split(" "))
splitRdd.take(5)


['EVERY', 'afternoon,', 'as', 'they', 'were']

In [4]:
# b

mappedRdd = splitRdd.map(lambda word: (word, 1))

mappedRdd.take(5)

[('EVERY', 1), ('afternoon,', 1), ('as', 1), ('they', 1), ('were', 1)]

In [5]:
## 3

# a

months = ("JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL")

monthsRdd = sc.parallelize(months)
monthsIndexed0Rdd = monthsRdd.zipWithIndex()

monthsIndexed0Rdd.collect()

[('JAN', 0),
 ('FEB', 1),
 ('MAR', 2),
 ('APR', 3),
 ('MAY', 4),
 ('JUN', 5),
 ('JUL', 6)]

In [6]:
# b

monthsIndexed1Rdd = monthsIndexed0Rdd.map(lambda item: (item[0], item[1]+1))
monthsIndexed1Rdd.collect()

[('JAN', 1),
 ('FEB', 2),
 ('MAR', 3),
 ('APR', 4),
 ('MAY', 5),
 ('JUN', 6),
 ('JUL', 7)]

In [7]:
# c
monthsIndexed2Rdd = monthsIndexed0Rdd.mapValues(lambda y: y+1)
monthsIndexed2Rdd.collect()

[('JAN', 1),
 ('FEB', 2),
 ('MAR', 3),
 ('APR', 4),
 ('MAY', 5),
 ('JUN', 6),
 ('JUL', 7)]

In [8]:
# d
quarters = (1, 1, 1, 2, 2, 2, 3)

quartersRdd = sc.parallelize(quarters)

monthsZipQuarters = monthsRdd.zip(quartersRdd)
monthsZipQuarters.collect()

[('JAN', 1),
 ('FEB', 1),
 ('MAR', 1),
 ('APR', 2),
 ('MAY', 2),
 ('JUN', 2),
 ('JUL', 3)]

In [9]:
# e
print(f"Keys => {monthsZipQuarters.keys().collect()}")
print(f"Values => {monthsZipQuarters.values().collect()}")
print(f"Sorted By keys => {monthsZipQuarters.sortByKey().collect()}")

Keys => ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL']
Values => [1, 1, 1, 2, 2, 2, 3]
Sorted By keys => [('APR', 2), ('FEB', 1), ('JAN', 1), ('JUL', 3), ('JUN', 2), ('MAR', 1), ('MAY', 2)]


In [10]:
### 4

# a
reducedByKeyRdd = mappedRdd.reduceByKey(lambda x, y: x + y)
reducedByKeyRdd.take(5)

[('EVERY', 1), ('as', 9), ('school,', 1), ('used', 4), ('go', 1)]

In [11]:
# b
flippedRdd = reducedByKeyRdd.map(lambda item: (item[1], item[0]))
flippedRdd.take(5)

[(1, 'EVERY'), (9, 'as'), (1, 'school,'), (4, 'used'), (1, 'go')]

In [12]:
# c
orderedRdd = flippedRdd.sortByKey(ascending=False)
orderedRdd.take(5)

[(148, 'the'), (85, 'and'), (44, 'he'), (38, 'to'), (33, '')]

In [13]:
# d

input_file = f"/home/talentum/Desktop/Assignments/Assignment1/Pair-RDD/flights.csv"

carrierRdd = sc.textFile(f"FILE://{input_file}").map(lambda line: line.split(",")).map(lambda cols: (cols[5], 1))
carrierRdd.take(5)


[('WN', 1), ('WN', 1), ('WN', 1), ('WN', 1), ('WN', 1)]

In [14]:
# e

carrierSorted = carrierRdd.reduceByKey(lambda x, y: x+y).map(lambda items: (items[1], items[0])).sortByKey(ascending=False)
carrierSorted.take(5)

[(356167, 'WN'),
 (175969, 'AA'),
 (166445, 'OO'),
 (141178, 'MQ'),
 (133403, 'US')]

In [15]:
### 2

# c

input_file = f"/home/talentum/Desktop/Assignments/Assignment1/Pair-RDD/airports.csv"

cityRdd = sc.textFile(f"FILE://{input_file}").map(lambda line: line.split(",")).map(lambda cols: (cols[0], cols[2]))
cityRdd.take(5)

[('iata', 'city'),
 ('00M', 'BaySprings'),
 ('00R', 'Livingston'),
 ('00V', 'ColoradoSprings'),
 ('01G', 'Perry')]

In [16]:
# d

input_file = f"/home/talentum/Desktop/Assignments/Assignment1/Pair-RDD/flights.csv"

flightOrigDestRdd = sc.textFile(f"FILE://{input_file}").map(lambda line: line.split(",")).map(lambda cols: (cols[12], cols[13]))
flightOrigDestRdd.take(5)

[('IAD', 'TPA'),
 ('IND', 'BWI'),
 ('IND', 'JAX'),
 ('IND', 'LAS'),
 ('IND', 'PHX')]

In [17]:
# e

origJoinRdd = flightOrigDestRdd.join(cityRdd)
origJoinRdd.take(5)

[('ONT', ('LAS', 'Ontario')),
 ('ONT', ('LAS', 'Ontario')),
 ('ONT', ('OAK', 'Ontario')),
 ('ONT', ('OAK', 'Ontario')),
 ('ONT', ('OAK', 'Ontario'))]

In [18]:
# f

destOrigJoinRdd = origJoinRdd.values().join(cityRdd)
destOrigJoinRdd.take(5)

[('LAS', ('Ontario', 'LasVegas')),
 ('LAS', ('Ontario', 'LasVegas')),
 ('LAS', ('Ontario', 'LasVegas')),
 ('LAS', ('Ontario', 'LasVegas')),
 ('LAS', ('Ontario', 'LasVegas'))]

In [19]:
# e

cityCleanedRdd = destOrigJoinRdd.values()
cityCleanedRdd.take(5)

[('Ontario', 'LasVegas'),
 ('Ontario', 'LasVegas'),
 ('Ontario', 'LasVegas'),
 ('Ontario', 'LasVegas'),
 ('Ontario', 'LasVegas')]

In [20]:
# h

citiesKV = cityCleanedRdd.map(lambda cities: (cities, 1))
citiesKV.take(5)

[(('Ontario', 'LasVegas'), 1),
 (('Ontario', 'LasVegas'), 1),
 (('Ontario', 'LasVegas'), 1),
 (('Ontario', 'LasVegas'), 1),
 (('Ontario', 'LasVegas'), 1)]

In [21]:
# i

citiesReducedSortedRdd = citiesKV.reduceByKey(lambda x, y: x + y).map(lambda items: (items[1], items[0])).sortByKey(ascending=False)
citiesReducedSortedRdd.take(3)

[(5540, ('NewYork', 'Boston')),
 (5478, ('Boston', 'NewYork')),
 (4103, ('Chicago', 'NewYork'))]

In [22]:
### 3

# b


input_file = f"/home/talentum/Desktop/Assignments/Assignment1/Pair-RDD/flights.csv"

delayRdd = sc.textFile(f"FILE://{input_file}").map(lambda line: line.split(",")).filter(lambda delay: int(delay[11]) > 15).map(lambda cols: (cols[5], cols[11]))


delayRdd.take(5)

[('WN', '25'), ('WN', '67'), ('WN', '87'), ('WN', '29'), ('WN', '82')]

In [23]:
# c

deplayMaxRdd = delayRdd.reduceByKey(lambda x, y: max(int(x), int(y)))
deplayMaxRdd.take(5)

[('XE', 781), ('YV', 526), ('OH', 680), ('OO', 767), ('UA', 1268)]

In [24]:
### 4

# c

input_file = f"/home/talentum/Desktop/Assignments/Assignment1/Pair-RDD/plane-data.csv"

planeDataRdd = sc.textFile(f"FILE://{input_file}")

planeDataRdd.count()

5030

In [25]:
# d

cleanedPlaneDataRdd = planeDataRdd.map(lambda val: val.split(",")).filter(lambda ele: len(ele) == 9)
cleanedPlaneDataRdd.count()

4481